### ЗАДАЧА: Пакетная загрузка отгрузок (try/except + custom exceptions)

Из внешней логистической системы приходят строки с отгрузками.
Нужно безопасно распарсить данные, отделить валидные записи от ошибок
и посчитать несколько итоговых метрик.

НЕОБХОДИМО РЕАЛИЗОВАТЬ:

1. Иерархию кастомных исключений:
   - `ShipmentError`
   - `RowFormatError`
   - `WeightError`
   - `PriorityError`
   - `RegionError`.

2. Функцию `parse_shipment(row)`:
   - формат строки: `shipment_id,client,weight,priority,region`
   - `weight` должен быть числом и `> 0`
   - допустимые приоритеты: `standard`, `express`, `vip`
   - допустимые регионы: `RU`, `KZ`, `BY`
   - при ошибке конвертации веса использовать `raise ... from ...`.

3. Функцию `load_shipments(rows)`:
   - вернуть `(shipments, errors)`
   - ошибки хранить как `(row, error_type, message)`
   - не останавливать цикл на первой ошибке.

4. Вывести:
   - число валидных отгрузок,
   - ошибки по типам,
   - суммарный вес только для `express` и `vip`,
   - клиента-лидера по суммарному весу среди валидных записей.

In [31]:
rows = [
    'S-100,Acme,12.5,express,RU',
    'S-101,Beta,0,standard,RU',
    'S-102,Acme,abc,vip,KZ',
    'S-103,Delta,8.5,urgent,BY',
    'S-104,Gamma,15,vip,UZ',
    'S-105,Acme,4.0,standard,KZ',
    'S-106,Beta,9.5,express,BY',
]


class ShipmentError(Exception):
    pass


class RowFormatError(ShipmentError):
    pass


class WeightError(ShipmentError):
    pass


class PriorityError(ShipmentError):
    pass


class RegionError(ShipmentError):
    pass


def parse_shipment(row):
    # TODO: распарсить строку и провалидировать weight, priority, region
    # TODO: при ошибке конвертации weight использовать raise ... from ...
    if len(row.split(",")) != 5:
        raise RowFormatError("Строка должна состоять из 5 элементов")
    
    shipment_id, client, weight, priority, region = row.split(",")
    try:
        weight = float(weight)
    except ValueError as e:
        raise WeightError("Вес не является числом") from e
    
    if weight < 0:
        raise WeightError("Вес не должен быть отрицательным")
    
    allowed_priorities = {"standard", "express", "vip"}
    if priority not in allowed_priorities:
        raise PriorityError("Недопустимый приоритет")
    
    allowed_regions = {"RU", "KZ", "BY"}
    if region not in allowed_regions:
        raise RegionError("Недопустимый регион")
    
    return {
        "shipment_id": shipment_id,
        "client": client,
        "weight": weight,
        "priority": priority,
        "region": region
    }
    

def load_shipments(rows):
    # TODO: вернуть (shipments, errors)
    shipments = []
    errors = []
    for row in rows:
        try:
            shipments.append(parse_shipment(row))
        except ShipmentError as e:
            errors.append((row, type(e).__name__, e))
    return shipments, errors

# TODO: вызвать load_shipments(rows)
shipments, errors = load_shipments(rows)

# TODO: вывести число валидных отгрузок и число ошибок
print(f"Количество валидных отгрузок = {len(shipments)} шт.")
print(f"Количество ошибочных отгрузок = {len(errors)} шт.")

# TODO: вывести ошибки по типам
type_errors = {}
for row, error, name in errors:
    if error not in type_errors:
        type_errors[error] = type_errors.get(error, 0) + 1
    print(f"Ошибка: '{error}', сообщение: '{name}', строка: '{row}'")
for error, count in type_errors.items():
    print(f"Ошибка '{error}' встречается {count} раз.")

# TODO: посчитать premium_weight только для express/vip
premium_weight = 0
for shipment in shipments:
    if shipment["priority"] == "express" and "vip":
        premium_weight += shipment["weight"]
print(f"Общий вес премиум отгрузок = {premium_weight} кг.")

# TODO: найти клиента-лидера по суммарному весу
max_count_client = {}
for shipment in shipments:
    client = shipment["client"]
    weight = shipment["weight"]
    if client not in max_count_client:
        max_count_client[client] = max_count_client.setdefault(client, 0)
    max_count_client[client] += weight

max_client = None
max_count = 0
for client, count in max_count_client.items():
    if count > max_count:
        max_client = client
        max_count = count
print(f"Клиент-лидер '{max_client}': суммарный вес отгрузок = {max_count} кг.")

Количество валидных отгрузок = 4 шт.
Количество ошибочных отгрузок = 3 шт.
Ошибка: 'WeightError', сообщение: 'Вес не является числом', строка: 'S-102,Acme,abc,vip,KZ'
Ошибка: 'PriorityError', сообщение: 'Недопустимый приоритет', строка: 'S-103,Delta,8.5,urgent,BY'
Ошибка: 'RegionError', сообщение: 'Недопустимый регион', строка: 'S-104,Gamma,15,vip,UZ'
Ошибка 'WeightError' встречается 1 раз.
Ошибка 'PriorityError' встречается 1 раз.
Ошибка 'RegionError' встречается 1 раз.
Общий вес премиум отгрузок = 22.0 кг.
Клиент-лидер 'Acme': суммарный вес отгрузок = 16.5 кг.
